In [ ]:
# ==============================================================================
# Cell 1: Package Installations
# ==============================================================================
!pip install -q groq neo4j sentence-transformers torch



In [ ]:
# ==============================================================================
# Cell 2: Imports & Environment Setup
# ==============================================================================
import os
import time
import json
from groq import Groq
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer

GROQ_API_KEY = "gsk_..."
NEO4J_URI = "bolt://127.0.0.1:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "password"

# Initialize Connections
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
embedder = SentenceTransformer("all-MiniLM-L6-v2")
client = Groq(api_key=GROQ_API_KEY)


In [ ]:

# ==============================================================================
# Cell 3: Database Index Setup & Document Ingestion
# ==============================================================================
with driver.session() as session:
    # Create HNSW Vector Index
    session.run("""
    CREATE VECTOR INDEX `document_embeddings` IF NOT EXISTS
    FOR (d:Document) ON (d.embedding)
    OPTIONS {indexConfig: {`vector.dimensions`: 384, `vector.similarity_function`: 'cosine'}}
    """)

    # Baseline Ingestion Data
    docs = [
        {"id": "doc1", "text": "GraphRAG combines vector search with knowledge graph traversals to give LLMs structured context.", "cat": "Architecture"},
        {"id": "doc2", "text": "Neo4j Community Edition supports native HNSW vector indexes without enterprise licensing.", "cat": "Database"}
    ]

    for doc in docs:
        vec = embedder.encode(doc["text"]).tolist()
        session.run("""
        MERGE (d:Document {id: $id})
        SET d.text = $text, d.embedding = $vec
        MERGE (c:Category {name: $cat})
        MERGE (d)-[:BELONGS_TO]->(c)
        """, id=doc["id"], text=doc["text"], vec=vec, cat=doc["cat"])

print("Neo4j database vector index ready & populated!")


In [ ]:

# ==============================================================================
# Cell 4: Define Self-Correcting Tools & Declarations
# ==============================================================================
def graph_cypher_traversal(entity_name: str) -> str:
    """Searches connected entities and categories in Neo4j."""
    try:
        with driver.session() as session:
            res = session.run("""
            MATCH (d:Document)-[:BELONGS_TO]->(c:Category)
            WHERE toLower(d.text) CONTAINS toLower($entity) OR toLower(c.name) CONTAINS toLower($entity)
            RETURN d.text AS doc, c.name AS category
            """, entity=entity_name)
            results = [f"Doc: {r['doc']} | Category: {r['category']}" for r in res]

            # SELF-CORRECTION HINT: Instructs agent to fallback to vector search if no matches
            if results:
                return "\n".join(results)
            else:
                return f"NO GRAPH MATCHES for '{entity_name}'. FALLBACK REQUIRED: Call 'vector_index_search' with a broader query for semantic retrieval."
    except Exception as e:
        return f"Database error: {str(e)}"

def vector_index_search(query: str) -> str:
    """Performs HNSW vector search to find matching document text in Neo4j."""
    try:
        query_vec = embedder.encode(query).tolist()
        with driver.session() as session:
            res = session.run("""
            CALL db.index.vector.queryNodes('document_embeddings', 2, $query_vec)
            YIELD node AS doc, score
            RETURN doc.text AS text, score
            """, query_vec=query_vec)
            results = [f"- {r['text']} (score: {round(r['score'], 2)})" for r in res]
            return "\n".join(results) if results else "No vector matches found."
    except Exception as e:
        return f"Database error: {str(e)}"

def python_calculator(expression: str) -> str:
    """Evaluates mathematical expressions deterministically."""
    try:
        allowed_chars = "0123456789+-*/(). "
        if all(c in allowed_chars for c in expression):
            return str(eval(expression))
        return "Error: Invalid characters in expression."
    except Exception as e:
        return f"Execution error: {str(e)}"

TOOL_MAP = {
    "vector_index_search": vector_index_search,
    "graph_cypher_traversal": graph_cypher_traversal,
    "python_calculator": python_calculator
}

groq_tools = [
    {
        "type": "function",
        "function": {
            "name": "vector_index_search",
            "description": "Performs HNSW vector search to find matching document text in Neo4j.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string", "description": "Search query text"}},
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "graph_cypher_traversal",
            "description": "Searches connected entities and categories in Neo4j.",
            "parameters": {
                "type": "object",
                "properties": {"entity_name": {"type": "string", "description": "Entity or category name to search"}},
                "required": ["entity_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "python_calculator",
            "description": "Evaluates math expressions safely.",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string", "description": "Math expression like '248.85 * 0.15'"}},
                "required": ["expression"]
            }
        }
    }
]


In [ ]:

# ==============================================================================
# Cell 5: Agentic Execution Loop with Fallback Guardrails
# ==============================================================================
def run_groq_agent_with_self_correction(user_goal: str, max_iterations: int = 4):
    print(f"User Goal: {user_goal}\n" + "="*50)

    system_prompt = """
    You are an AI Agent with tool access. Follow these self-correction rules:
    1. If 'graph_cypher_traversal' returns no matches or a FALLBACK REQUIRED message, DO NOT give up.
    2. Immediately call 'vector_index_search' using relevant keywords as a secondary retrieval strategy.
    3. Synthesize a final answer once all fallback options have been evaluated.
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_goal}
    ]

    for iteration in range(1, max_iterations + 1):
        print(f"\n[Iteration {iteration}] Agent Thinking...")

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            tools=groq_tools,
            tool_choice="auto"
        )

        response_message = response.choices[0].message
        tool_calls = response_message.tool_calls

        if tool_calls:
            messages.append(response_message)
            for tool_call in tool_calls:
                fn_name = tool_call.function.name
                fn_args = json.loads(tool_call.function.arguments)
                print(f"Agent Action: Executing Tool `{fn_name}` with args: {fn_args}")

                if fn_name in TOOL_MAP:
                    tool_output = TOOL_MAP[fn_name](**fn_args)
                    print(f"Tool Output:\n{tool_output}")
                    messages.append({
                        "tool_call_id": tool_call.id,
                        "role": "tool",
                        "name": fn_name,
                        "content": tool_output
                    })
        else:
            print("\nFinal Answer Generated:")
            print("-" * 50)
            print(response_message.content.strip())
            return response_message.content.strip()

# ==============================================================================
# Cell 6: Run Self-Correcting Test
# ==============================================================================
run_groq_agent_with_self_correction("Search the knowledge graph for Neo4j features and calculate 15% of 248.85s latency.")